In [1]:
import sys, os
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df, convert_col
import pickle

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

In [3]:
%LoadCheckpoint /scratch/jieq/pandax/dias_notebooks/joshuaswords/netflix-data-visualization/src/rewritten/o4_mini_high_small/checkpoints/post_cell_12_try_3.pickle

trying: ['df_cell6']


me:  18
trying: ['y']
me:  14
trying: ['r']
me:  14
trying: ['data']
me:  20
trying: ['country_order']
me:  22
trying: ['warnings']
me:  1
trying: ['data_q2q3']
me:  22
trying: ['np']
me:  0
trying: ['data_q2q3_ratio']
me:  22
trying: ['BENCHMARKS_TO_PATHS']
me:  0
trying: ['Path']
me:  0
trying: ['pd']
me:  0
trying: ['benchmark_name']
me:  2
trying: ['factor']
me:  2
trying: ['mf']
me:  26
trying: ['null_rate']
me:  4
trying: ['i']
me:  4
trying: ['rating_order']
me:  24
trying: ['order']
me:  24
trying: ['mf_ratio']
me:  14
trying: ['i_1']
me:  16
trying: ['orig_output']
me:  15
trying: ['i_2']
me:  16
trying: ['df']
me:  3
trying: ['movie']
me:  26
trying: ['tv']
me:  26
trying: ['x']
me:  14
trying: ['ratings_ages']
me:  18
Declaring variable np
Declaring variable BENCHMARKS_TO_PATHS
Declaring variable Path
Declaring variable pd
Declaring variable warnings
Declaring variable benchmark_name
Declaring variable factor
Declaring variable df
Declaring variable null_rate
Declaring varia

In [4]:
%%RecordEvent
%%time
### cell 13 ###

# Optimized for cudf: perform grouping and counting on GPU
# Remove CPU-based value_counts by grouping on both columns and using size()
data_sub = (
    df_cell6
    .groupby(["type", "year_added"])      # GPU-enabled groupby on both columns
    .size()                                  # GPU-enabled count of each combination
    .unstack()                               # GPU-enabled pivot: types as rows, years as columns
    .fillna(0)                               # fill missing counts with 0 on GPU
    .loc[["TV Show", "Movie"]]           # select only the two types for stacking
    .cumsum(axis=0)                          # compute cumulative sums across types (for stacked area)
    .T                                       # transpose: years as index, types as columns
)

CPU times: user 492 ms, sys: 182 ms, total: 673 ms
Wall time: 651 ms


In [5]:
%Checkpoint /scratch/jieq/pandax/dias_notebooks/joshuaswords/netflix-data-visualization/src/rewritten/o4_mini_high_small/checkpoints/post_cell_13_try_0.pickle

migration speed (bps): 875715010.9389086


---------------------------
variables to migrate:
orig_output 16
data_sub 336
ratings_ages 640
df 9352096
data_q2q3_ratio 284
data_q2q3 372
country_order 124
np 72
pd 72
null_rate 32
i 60
rating_order 184
tv 170
movie 170
warnings 72
r 40
mf 136
y 28
mf_ratio 28
i_1 53
i_2 53
BENCHMARKS_TO_PATHS 2272
Path 904
factor 28
benchmark_name 75
order 226
x 40
df_cell6 595448600
data 179
---------------------------
variables to recompute:
[]
---------------------------
cells to recompute:
[]
Checkpoint saved to: /scratch/jieq/pandax/dias_notebooks/joshuaswords/netflix-data-visualization/src/rewritten/o4_mini_high_small/checkpoints/post_cell_13_try_0.pickle


In [6]:
%PrintCellInfo opt_cell_exec_info

======= Cell 0 =======
Input variables ['BENCHMARKS_TO_PATHS', 'Path']
Active variables ['df', 'df_cell6']
Intermediate variables ['benchmark_name', 'factor']
Future variables []
Modified dataframes
======= Cell 1 =======
Input variables ['df_cell6']
Active variables []
Intermediate variables ['i', 'null_rate']
Future variables []
Modified dataframes
======= Cell 2 =======
Input variables ['df_cell6']
Active variables ['df_cell6']
Intermediate variables []
Future variables []
Modified dataframes
  df_cell6
    Input columns: set()
    Changed columns: {'date_added', 'country', 'cast', 'release_year', 'description', 'listed_in', 'duration', 'title', 'rating', 'type', 'director', 'show_id'}
    Created columns: set()
    Deleted columns: set()
======= Cell 3 =======
Input variables ['df_cell6']
Active variables []
Intermediate variables []
Future variables ['df_cell6']
Modified dataframes
======= Cell 4 =======
Input variables ['df_cell6']
Active variables []
Intermediate variables []
Fu

In [7]:

with open("/scratch/jieq/pandax/dias_notebooks/joshuaswords/netflix-data-visualization/src/opt_cell_exec_info_13_try_0.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[13], f)


In [8]:
opt_output = Out.get(4)

In [9]:
data_sub_opt = data_sub
%LoadCheckpoint /scratch/jieq/pandax/dias_notebooks/joshuaswords/netflix-data-visualization/src/small_bench/checkpoints/post_cell_13.pickle
assert compare_df(data_sub_opt, data_sub)

import numpy as np
if os.getenv("USE_GPU") == "True":
    import cudf
from elastic.core.common.pandas import is_type_styler
is_orig_output_pd = isinstance(orig_output, (pd.Series, pd.DataFrame, pd.Index))
is_opt_output_pd = isinstance(opt_output, (pd.Series, pd.DataFrame, pd.Index))
if os.getenv("USE_GPU") == "True":
    is_orig_output_array = isinstance(orig_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
    is_opt_output_array = isinstance(opt_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
else:
    is_orig_output_array = isinstance(orig_output, np.ndarray)
    is_opt_output_array = isinstance(opt_output, np.ndarray)

is_orig_output_styler = is_type_styler(type(orig_output))
is_opt_output_styler = is_type_styler(type(opt_output))
if is_orig_output_styler and is_opt_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_orig_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_opt_output_styler:
    assert opt_output.to_html() == orig_output

if is_orig_output_pd and is_opt_output_pd:
    assert orig_output.equals(opt_output)
# TODO(jie): this is a hack.
elif ((is_orig_output_pd or is_opt_output_pd) and (is_orig_output_array or is_opt_output_array)) or (is_orig_output_array and is_opt_output_array):
    assert list(orig_output) == list(opt_output)
else:
    assert orig_output == opt_output


trying: ['order']
me:  24
trying: ['i']
me:  4
trying: ['ratings_ages']
me:  18
trying: ['null_rate']
me:  4
trying: ['df_cell6']


me:  18
trying: ['data']


me:  20
trying: ['country_order']
me:  22
trying: ['warnings']
me:  1
trying: ['data_q2q3_ratio']
me:  22
trying: ['df']
me:  3
trying: ['data_q2q3']
me:  22
trying: ['BENCHMARKS_TO_PATHS']
me:  0
trying: ['r']
me:  14
trying: ['np']
me:  0
trying: ['pd']
me:  0
trying: ['Path']
me:  0
trying: ['rating_order']
me:  24
trying: ['mf']
me:  26
trying: ['y']
me:  14
trying: ['tv']
me:  26
trying: ['mf_ratio']
me:  14
trying: ['movie']
me:  26
trying: ['orig_output']
me:  15
trying: ['i_1']
me:  16
trying: ['i_2']
me:  16
trying: ['benchmark_name']
me:  2
trying: ['factor']
me:  2
trying: ['data_sub']
me:  28
trying: ['x']
me:  14
Declaring variable BENCHMARKS_TO_PATHS
Declaring variable np
Declaring variable pd
Declaring variable Path
Declaring variable warnings
Declaring variable benchmark_name
Declaring variable factor
Declaring variable df
Declaring variable i
Declaring variable null_rate
Declaring variable r
Declaring variable y
Declaring variable mf_ratio
Declaring variable x
Declarin